# FINDETECT Block D — Partitioned staging

This is the finalized staging step for the Block D numerical feature pipeline.
It partitions the upstream Block B (`B.parquet`) and Block C (`C.parquet`) outputs
into 64 buckets and writes three datasets: `sender_transactions`,
`incoming_events`, and `behavior`. The chosen partitioning lets the numerical
feature notebook process one bucket at a time while preserving the history
needed for each sender.

**Execution:** in Google Colab, run the cells in order and then run
`02_Block_D_Numerical_Features.ipynb`. The upstream Parquet files and generated
partitions are external data; they are not embedded in the GitHub notebook.
Completed staging datasets are reused on a rerun.


In [ ]:
%pip install -q duckdb
from google.colab import drive
drive.mount('/content/drive')

## 1. Define staging and input validation functions


In [ ]:
"""Efficient two-stage FINDETECT Block D pipeline for Colab."""
from __future__ import annotations
import json, shutil, time
from pathlib import Path
import duckdb

LABEL_COLUMNS = ["Is Laundering", "Pattern Type"]

def qi(x): return '"' + str(x).replace('"', '""') + '"'
def qs(x): return "'" + str(x).replace("'", "''") + "'"
def pq(x): return f"read_parquet({qs(str(x))})"

def connect(memory_limit="6GB", threads=2, temp_dir="/content/findetect_tmp"):
    Path(temp_dir).mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.execute(f"SET memory_limit={qs(memory_limit)}")
    con.execute(f"SET threads={int(threads)}")
    con.execute(f"SET temp_directory={qs(temp_dir)}")
    con.execute("SET preserve_insertion_order=false")
    return con

def cols(con, path):
    return {r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {pq(path)}").fetchall()}

def validate_inputs(b_path, c_path, memory_limit="6GB", threads=2):
    con = connect(memory_limit, threads, "/content/findetect_validate")
    bc, cc = cols(con,b_path), cols(con,c_path)
    need_b={"Final_Account","Week","total_amount_sent_prev_week","avg_amount_sent_prev_week",
      "max_amount_sent_prev_week","outgoing_transaction_count","unique_receivers",
      "total_amount_received_prev_week","avg_amount_received_prev_week",
      "max_amount_received_prev_week","incoming_transaction_count","unique_senders"}
    need_c={"Timestamp","Account","Account_duplicated_0","Amount Paid","Amount Received",
      "Payment Format","Week","Week_Index","Graph_History_Week_Index","sender_out_degree",
      "sender_in_degree","sender_unique_counterparties","receiver_out_degree","receiver_in_degree",
      "receiver_unique_counterparties","sender_receiver_frequency","bidirectional_relationship",*LABEL_COLUMNS}
    if need_b-bc: raise ValueError(f"B missing: {sorted(need_b-bc)}")
    if need_c-cc: raise ValueError(f"C missing: {sorted(need_c-cc)}")
    dup=con.execute(f"SELECT count(*) FROM (SELECT {qi('Final_Account')},{qi('Week')} FROM {pq(b_path)} GROUP BY ALL HAVING count(*)>1)").fetchone()[0]
    if dup: raise ValueError(f"B has {dup:,} duplicate account-week keys")
    bad=con.execute(f"SELECT count(*) FROM {pq(c_path)} WHERE {qi('Graph_History_Week_Index')} IS NOT NULL AND {qi('Graph_History_Week_Index')}<>{qi('Week_Index')}-1").fetchone()[0]
    if bad: raise ValueError(f"Graph history check failed on {bad:,} rows")
    nr=con.execute(f"SELECT count(*) FROM {pq(c_path)}").fetchone()[0]
    con.close(); return {"source_rows":nr,"c_columns":cc}

def _safe_stage_dir(path):
    p=Path(path)
    if "findetect" not in str(p).lower() or len(p.parts)<4:
        raise ValueError(f"Unsafe staging path: {p}")
    return p

def _stage_one(name, query, stage_root, expected_rows, memory_limit, threads):
    root=_safe_stage_dir(stage_root); target=root/name; marker=target/"_SUCCESS.json"
    if marker.exists():
        meta=json.loads(marker.read_text())
        if meta.get("expected_rows")==expected_rows:
            print(f"{name}: already staged — skipped"); return meta
    if target.exists(): shutil.rmtree(target)
    target.mkdir(parents=True)
    con=connect(memory_limit,threads,f"/content/findetect_stage_tmp/{name}")
    started=time.time()
    print(f"{name}: partitioning once...")
    con.execute(f"COPY ({query}) TO {qs(target)} (FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY(d_bucket_id), ROW_GROUP_SIZE 100000)")
    con.close()
    data_files=list(target.glob("d_bucket_id=*/*.parquet"))
    if not data_files: raise RuntimeError(f"{name}: no partition files created")
    meta={"name":name,"expected_rows":expected_rows,"files":len(data_files),"minutes":round((time.time()-started)/60,2)}
    marker.write_text(json.dumps(meta,indent=2)); print(f"{name}: complete in {meta['minutes']} min")
    return meta

def stage_inputs(b_path,c_path,stage_root,total_buckets,c_columns,memory_limit="6GB",threads=2):
    con=connect(memory_limit,threads,"/content/findetect_stage_counts")
    c_rows=con.execute(f"SELECT count(*) FROM {pq(c_path)}").fetchone()[0]
    incoming_rows=con.execute(f"SELECT count(*) FROM {pq(c_path)} WHERE {qi('Timestamp')} IS NOT NULL AND {qi('Account_duplicated_0')} IS NOT NULL").fetchone()[0]
    b_rows=con.execute(f"SELECT count(*) FROM {pq(b_path)}").fetchone()[0]; con.close()
    safe="* EXCLUDE (Pattern_Type_Encoded)" if "Pattern_Type_Encoded" in c_columns else "*"
    sender_q=f"SELECT {safe}, mod(hash(coalesce({qi('Account')}::VARCHAR,'')),{total_buckets})::INT d_bucket_id FROM {pq(c_path)}"
    incoming_q=f"SELECT {qi('Account_duplicated_0')} incoming_account,{qi('Timestamp')} incoming_ts,{qi('Amount Received')}::DOUBLE incoming_amount, mod(hash(coalesce({qi('Account_duplicated_0')}::VARCHAR,'')),{total_buckets})::INT d_bucket_id FROM {pq(c_path)} WHERE {qi('Timestamp')} IS NOT NULL AND {qi('Account_duplicated_0')} IS NOT NULL"
    b_q=f"SELECT cur.*,prev.{qi('outgoing_transaction_count')} outgoing_count_prev_week,prev.{qi('incoming_transaction_count')} incoming_count_prev_week,mod(hash(coalesce(cur.{qi('Final_Account')}::VARCHAR,'')),{total_buckets})::INT d_bucket_id FROM {pq(b_path)} cur LEFT JOIN {pq(b_path)} prev ON cur.{qi('Final_Account')}=prev.{qi('Final_Account')} AND cur.{qi('Week')}=prev.{qi('Week')}+INTERVAL 7 DAY"
    return {
      "sender":_stage_one("sender_transactions",sender_q,stage_root,c_rows,memory_limit,threads),
      "incoming":_stage_one("incoming_events",incoming_q,stage_root,incoming_rows,memory_limit,threads),
      "behavior":_stage_one("behavior",b_q,stage_root,b_rows,memory_limit,threads)}


## 2. Input and output paths

`B_PATH` and `C_PATH` point to the finalized upstream Block B and Block C files.
`STAGE_ROOT` is the partitioned output consumed by the next notebook. Set the
paths for your environment; keep `TOTAL_BUCKETS=64` for this pipeline.


In [ ]:
B_PATH='/content/drive/MyDrive/processed/B.parquet'
C_PATH='/content/drive/MyDrive/processed/C.parquet'
STAGE_ROOT='/content/drive/MyDrive/processed/findetect_D_stage_v2'
TOTAL_BUCKETS=64
MEMORY_LIMIT='6GB'
THREADS=2

## 3. Validate the input schemas and row count


In [ ]:
check=validate_inputs(B_PATH,C_PATH,MEMORY_LIMIT,THREADS)
print(f"C rows: {check['source_rows']:,}")
print('INPUT VALIDATION PASSED')

## 4. Generate the three staging datasets

This step writes the finalized sender, incoming, and behavior partitions under
`STAGE_ROOT`. If execution is interrupted, rerun the cells in order; datasets
with completed row-count markers are reused.


In [ ]:
stage_result=stage_inputs(B_PATH,C_PATH,STAGE_ROOT,TOTAL_BUCKETS,check['c_columns'],MEMORY_LIMIT,THREADS)
print(json.dumps(stage_result,indent=2))